In [2]:
import json

with open('/Users/andrejpihtin/учеба/комп_мага/APPLIED_NN/project/ChatExport_2026-06-24 (2)/result.json', 'r') as f: # open in readonly mode
    obj = json.load(f)

In [9]:
messages = obj['messages']

In [15]:
len(messages), type(messages)

(88531, list)

In [1]:
from pymongo import MongoClient, IndexModel, ASCENDING

In [2]:
# Подключаемся к MongoDB (локальному серверу по умолчанию)
client = MongoClient("mongodb://localhost:27017/")

# Получаем (или создаём при первом использовании) базу данных
db = client["messages_me&masha"]   # ← база данных ещё не создана на диске

In [3]:
# Получаем (или создаём) коллекцию
collection = db["messages_collection"]          # ← тоже пока только в памяти

# result = collection.insert_many(messages) # Включать только для загрузки данных

# print("Inserted IDs:", result.inserted_ids)


In [4]:
# Задаем запрос, нужна только объекты с типом message и от меня
query = {"type": "message", 
         "from": 'Андрей Пихтин', 
         "text": {"$ne": ""}, 
         "forwarded_from": {"$exists": False}
         }

In [27]:
from datetime import datetime

for doc in collection.find({"date": {"$type": "string"}}):
    dt = datetime.fromisoformat(doc["date"])
    collection.update_one(
        {"_id": doc["_id"]},
        {"$set": {"date": dt}}
    )

In [28]:
collection.find_one()

{'_id': ObjectId('6a3c0fc988b59a6dbb3940a7'),
 'id': 6549,
 'type': 'service',
 'date': datetime.datetime(2019, 5, 15, 22, 38, 40),
 'date_unixtime': '1557949120',
 'actor': 'Маша',
 'actor_id': 'user851744305',
 'action': 'joined_telegram',
 'text': '',
 'text_entities': []}

In [34]:
conv_query = {"type": "message", "text": {"$ne": ""}, "forwarded_from": {"$exists": False}}

In [35]:
conversations = collection.find(conv_query).sort("date", 1)

dialogues = []
current = []
prev_date = None

for msg in conversations:
    if prev_date and (msg["date"] - prev_date).total_seconds() > 1800:  # 30 minutes
        dialogues.append(current)
        current = []

    current.append(msg)
    prev_date = msg["date"]

if current:
    dialogues.append(current)

In [36]:
type(dialogues), len(dialogues)

(list, 5431)

In [39]:
def filter_interaction_groups(groups):
    """
    Сохраняю только те диалоги, где есть более одного участника
    """
    filtered = []

    for group in groups:
        senders = {msg.get("from_id") or msg.get("from") for msg in group}

        if len(senders) > 1:
            filtered.append(group)

    return filtered

In [40]:
filtered = filter_interaction_groups(dialogues)

In [41]:
len(filtered)

3786

In [43]:
filtered[0]

[{'_id': ObjectId('6a3c0fc988b59a6dbb3940cb'),
  'id': 13117,
  'type': 'message',
  'date': datetime.datetime(2020, 3, 22, 0, 56, 58),
  'date_unixtime': '1584827818',
  'from': 'Андрей Пихтин',
  'from_id': 'user424591162',
  'text': 'Вторая попытка',
  'text_entities': [{'type': 'plain', 'text': 'Вторая попытка'}]},
 {'_id': ObjectId('6a3c0fc988b59a6dbb3940cd'),
  'id': 13119,
  'type': 'message',
  'date': datetime.datetime(2020, 3, 22, 0, 58, 37),
  'date_unixtime': '1584827917',
  'from': 'Маша',
  'from_id': 'user851744305',
  'text': 'подожди',
  'text_entities': [{'type': 'plain', 'text': 'подожди'}]},
 {'_id': ObjectId('6a3c0fc988b59a6dbb3940ce'),
  'id': 13120,
  'type': 'message',
  'date': datetime.datetime(2020, 3, 22, 0, 58, 39),
  'date_unixtime': '1584827919',
  'from': 'Маша',
  'from_id': 'user851744305',
  'text': 'это саша?????',
  'text_entities': [{'type': 'plain', 'text': 'это саша?????'}]},
 {'_id': ObjectId('6a3c0fc988b59a6dbb3940cf'),
  'id': 13121,
  'type':

In [42]:
def to_llm_messages(messages, user_name="Маша"):
    llm_messages = []

    for msg in messages:
        text = msg.get("text")

        # handle Telegram-style "text" which can be list/dict
        if isinstance(text, list):
            text = "".join(
                part["text"] if isinstance(part, dict) else str(part)
                for part in text
            )

        role = "user" if msg.get("from") == user_name else "assistant"

        llm_messages.append({
            "role": role,
            "content": text
        })

    return llm_messages

In [54]:
dataset = []
for conv in filtered:
    dataset.append(to_llm_messages(conv))

In [63]:
consecutive_dialogues = []
for conv in dataset:
    new_conv = []
    first_msg = conv[0]
    current_role = first_msg["role"]
    if current_role == 'assistant':
        continue
    merged_stack = [first_msg['content']]
    for msg in conv[1:]:
        if msg["role"] == current_role:
            merged_stack.append(msg['content'])
        else:
            new_conv.append(
                {
                    'role': current_role,
                    'content': '/n'.join(merged_stack)
                }  
                )
            current_role = msg["role"]
            merged_stack = [msg['content']]
    if len(new_conv) > 1:
        consecutive_dialogues.append(new_conv)


In [64]:
len(consecutive_dialogues) # Сделал просто, отрезал все диалоги, которые начинаются с ассистента. Потерял много, но 2 это хорошее количество примеров.

1368

In [ ]:
import json

with open('/Users/andrejpihtin/учеба/комп_мага/APPLIED_NN/project/dataset.json', 'w')as f:
    f.write(json.dumps(consecutive_dialogues, ensure_ascii=False))